> **FES · Fonaments d'Enginyeria de Software amb Python**

# 14 · SQLite i històrics de sensors

**Ampliació opcional.** Dedicació addicional orientativa d'1 hora de teoria i
1 hora de pràctica, fora de les 14 hores obligatòries.

## Objectius d'aprenentatge

- Crear una taula i inserir, consultar, modificar i eliminar lectures.
- Utilitzar consultes parametritzades amb `?`.
- Explicar `commit`, `rollback` i el tancament d'una connexió.

El sistema de sensors conservarà un històric en una base de dades local.
SQLite forma part de Python a través de `sqlite3` i funciona sense servidor.
Cal conèixer funcions, tuples, fitxers i excepcions.

## Connexió, taula i recursos

`sqlite3.connect(ruta)` obre o crea una base de dades en un fitxer.
Una taula agrupa files i columnes. `CREATE TABLE` defineix la seva estructura.
`INTEGER`, `TEXT` i `REAL` indiquen tipus SQL. `PRIMARY KEY` identifica cada
fila i `NOT NULL` exigeix que hi hagi un valor.

`execute` envia una instrucció a SQLite. `commit` confirma canvis.
`closing`, de `contextlib`, tanca la connexió quan acaba el bloc.
El context propi d'una connexió gestiona transaccions i cal tancar-la
explícitament. Utilitzarem `closing` per fer aquest tancament.

Les demostracions creen bases dins de carpetes temporals que es netegen en
acabar. Les funcions reben la ruta perquè es puguin reutilitzar.

In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory


def crear_historic(ruta):
    """Crea la taula si encara no existeix."""
    with closing(sqlite3.connect(ruta)) as connexio:
        connexio.execute("""
            CREATE TABLE IF NOT EXISTS lectures (
                id INTEGER PRIMARY KEY,
                sensor TEXT NOT NULL,
                valor REAL NOT NULL,
                unitat TEXT NOT NULL
            )
        """)
        connexio.commit()


with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "historic.sqlite"
    crear_historic(ruta)
    print(ruta.exists())  # True

## Inserir dades i recuperar-les

`INSERT INTO` afegeix una fila. Cada `?` representa un valor rebut per separat
en una tupla. Els paràmetres tracten correctament noms amb apòstrofs.
En una taula amb `INTEGER PRIMARY KEY`, SQLite assigna un id si l'ometem.

`SELECT` recupera columnes. `ORDER BY id` fixa l'ordre de les files.
`execute` retorna un cursor i `fetchall()` en recupera una llista de tuples.
Una taula buida retorna `[]`. `cursor.lastrowid` és l'identificador assignat
a l'última fila inserida. La funció el retorna per poder-lo utilitzar.

Prediu si les lectures es conservaran quan tanquem una connexió i n'obrim una
al mateix fitxer.

In [ ]:
def afegir_lectura(ruta, sensor, valor, unitat):
    """Insereix una lectura amb dades parametritzades."""
    with closing(sqlite3.connect(ruta)) as connexio:
        cursor = connexio.execute(
            "INSERT INTO lectures (sensor, valor, unitat) "
            "VALUES (?, ?, ?)",
            (sensor, valor, unitat),
        )
        connexio.commit()
        return cursor.lastrowid


def llegir_historic(ruta):
    """Retorna tuples ordenades per identificador."""
    with closing(sqlite3.connect(ruta)) as connexio:
        cursor = connexio.execute(
            "SELECT id, sensor, valor, unitat FROM lectures ORDER BY id"
        )
        return cursor.fetchall()


with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "historic.sqlite"
    crear_historic(ruta)
    print(llegir_historic(ruta))  # []
    print(afegir_lectura(ruta, "Sensor d'entrada", 3.3, "V"))  # 1
    afegir_lectura(ruta, "T1", 21.5, "°C")
    print(llegir_historic(ruta))

## Consultar un sensor amb WHERE

`WHERE sensor = ?` restringeix la consulta al nom indicat.
La tupla `(nom,)` té un únic element. La coma és necessària.

Aquesta consulta forma part del sistema de monitoratge i recupera les mesures
d'un sensor al llarg del temps. Un nom desconegut retorna una llista buida.
Les dades sempre viatgen com a paràmetres separats del text SQL.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "historic.sqlite"
    crear_historic(ruta)
    afegir_lectura(ruta, "Sensor d'entrada", 3.3, "V")
    afegir_lectura(ruta, "Sensor d'entrada", 3.4, "V")
    with closing(sqlite3.connect(ruta)) as connexio:
        for nom in ["Sensor d'entrada", "desconegut"]:
            cursor = connexio.execute(
                "SELECT valor, unitat FROM lectures "
                "WHERE sensor = ? ORDER BY id",
                (nom,),
            )
            print(cursor.fetchall())
    # [(3.3, 'V'), (3.4, 'V')]
    # []

## Modificar i eliminar files

`UPDATE` modifica columnes i `DELETE` elimina files. El filtre `WHERE id = ?`
limita l'operació a una lectura. `cursor.rowcount` indica quantes files s'han
afectat. Un identificador absent dona zero.

Les dues modificacions següents es confirmen amb un sol `commit`. Una
transacció agrupa canvis que han de confirmar-se conjuntament.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "historic.sqlite"
    crear_historic(ruta)
    afegir_lectura(ruta, "Sensor d'entrada", 3.3, "V")
    afegir_lectura(ruta, "T1", 21.5, "°C")
    with closing(sqlite3.connect(ruta)) as connexio:
        cursor = connexio.execute(
            "UPDATE lectures SET valor = ? WHERE id = ?", (3.5, 1)
        )
        print(cursor.rowcount)  # 1
        cursor = connexio.execute("DELETE FROM lectures WHERE id = ?", (2,))
        print(cursor.rowcount)  # 1
        cursor = connexio.execute("DELETE FROM lectures WHERE id = ?", (999,))
        print(cursor.rowcount)  # 0
        connexio.commit()
    print(llegir_historic(ruta))  # [(1, "Sensor d'entrada", 3.5, 'V')]

## Desfer un lot quan falla

`rollback` desfà els canvis pendents de la transacció.
La funció següent confirma un lot quan totes les insercions acaben bé.
Si SQLite genera una excepció, desfà el lot i propaga l'error amb `raise`.

Enviar `None` al camp `sensor` produeix `NULL` i incompleix `NOT NULL`.
SQLite genera `sqlite3.IntegrityError`, que és una subclasse de
`sqlite3.Error`. Provarem el rebuig sobre una taula buida i sobre una taula
amb dades, i comprovarem que el seu estat es conserva.

In [ ]:
def afegir_lot(ruta, lectures):
    """Confirma tot el lot o el desfà davant d'un error SQLite."""
    with closing(sqlite3.connect(ruta)) as connexio:
        try:
            for sensor, valor, unitat in lectures:
                connexio.execute(
                    "INSERT INTO lectures (sensor, valor, unitat) "
                    "VALUES (?, ?, ?)",
                    (sensor, valor, unitat),
                )
            connexio.commit()
        except sqlite3.Error:
            connexio.rollback()
            raise


with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "historic.sqlite"
    crear_historic(ruta)
    lot_invalid = [("H2", 50.0, "%"), (None, 51.0, "%")]
    for lot_previ in [[], [("H1", 45.0, "%"), ("H1", 46.0, "%")]]:
        afegir_lot(ruta, lot_previ)
        anterior = llegir_historic(ruta)
        try:
            afegir_lot(ruta, lot_invalid)
        except sqlite3.IntegrityError:
            print("Lot rebutjat i desfet")
        assert llegir_historic(ruta) == anterior
        print(llegir_historic(ruta))
    # Primer es conserva []; després es conserven les dues lectures H1.

## Pràctica i projecte evolutiu

Resol els quatre [exercicis](01_sqlite_exercicis.py). Els dos primers
practiquen la creació i la consulta, el tercer tracta les modificacions i
el quart és un repte de transaccions. Compara els resultats amb les
[solucions](01_sqlite_solucions.py).

Tot aquest bloc és ampliació opcional. El sistema dels blocs 08–13 ja funciona
amb CSV i JSON. Aquesta sessió afegeix un històric relacional i consultes per
sensor. Els camps conserven els noms `sensor`, `valor` i `unitat`.

La base de dades conserva les dades entre connexions fins que eliminem el
fitxer. Les carpetes temporals permeten verificar les operacions amb dades
de prova i retirar-les en acabar.